# 🔑 Solusi — CNN Mini dari Nol (Bab 5)> **Buka HANYA SETELAH** mengerjakan `starter.ipynb` + semua test hijau.Cell di notebook ini memakai implementasi referensi (`solusi/cnn_mini_ref.py`)supaya bisa dijalankan meskipun stub `nn_mini` kamu masih TODO — angkanyajadi pembanding untuk eksperimenmu.

## Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd() / 'solusi'))

import matplotlib.pyplot as plt
import numpy as np

import cnn_mini_ref as ref
from nn_mini.data import NILAI_KELAS, muat_semua
from nn_mini.nnet import Dense, Sequential
from nn_mini.train import train_model

X_train, y_train, X_val, y_val, X_test, y_test = muat_semua()
print('splits:', len(y_train), len(y_val), len(y_test))


## Baseline MLP (piksel mentah)

In [ ]:
Xtr = X_train.reshape(len(X_train), -1)
Xva = X_val.reshape(len(X_val), -1)

mlp = Sequential([Dense(64, activation='relu', seed=1),
                  Dense(16, activation='relu', seed=2),
                  Dense(1, activation='sigmoid', seed=3)])
hist_mlp = train_model(mlp, Xtr, y_train, Xva, y_val,
                       epochs=60, batch_size=32, lr=0.3, seed=0)
print(f'MLP: val acc = {hist_mlp["val_acc"][-1]:.4f}')


## CNN Mini (implementasi referensi)

In [ ]:
cnn = ref.CNNMini(seed=0)
hist_cnn = train_model(cnn, X_train, y_train, X_val, y_val,
                       epochs=30, batch_size=32, lr=0.5, seed=0)
print(f'CNN: val acc = {hist_cnn["val_acc"][-1]:.4f}, '
      f'val loss = {hist_cnn["val_loss"][-1]:.4f}')

plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
plt.plot(hist_mlp['train_loss'], label='MLP train')
plt.plot(hist_mlp['val_loss'], '--', label='MLP val')
plt.plot(hist_cnn['train_loss'], label='CNN train')
plt.plot(hist_cnn['val_loss'], '--', label='CNN val')
plt.xlabel('epoch'); plt.ylabel('BCE'); plt.title('Loss'); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(hist_mlp['val_acc'], label='MLP')
plt.plot(hist_cnn['val_acc'], label='CNN')
plt.xlabel('epoch'); plt.ylabel('val accuracy'); plt.title('Akurasi'); plt.legend()
plt.tight_layout(); plt.show()


## Filter yang dipelajari

In [ ]:
K = cnn.conv.K
print('Filter 3x3 hasil training:\n', np.round(K, 2))

from nn_mini.conv import conv2d_forward, maxpool2x2_forward, pad_zero

fig, axes = plt.subplots(2, 4, figsize=(13, 6))
for k, kelas in enumerate([0, 1]):
    img = X_val[y_val == kelas][0]
    resp = conv2d_forward(pad_zero(img[None], 1), K)[0]
    axes[k, 0].imshow(img, cmap='gray'); axes[k, 0].set_title('input')
    axes[k, 1].imshow(resp, cmap='viridis'); axes[k, 1].set_title('setelah conv')
    axes[k, 2].imshow(np.maximum(resp, 0), cmap='viridis'); axes[k, 2].set_title('setelah ReLU')
    axes[k, 3].imshow(maxpool2x2_forward(np.maximum(resp, 0)[None])[0], cmap='viridis')
    axes[k, 3].set_title('setelah max-pool')
    for j in range(4):
        axes[k, j].axis('off')
plt.suptitle('Aliran informasi: conv → ReLU → pool')
plt.tight_layout(); plt.show()


## Eksperimen ukuran kernel

In [ ]:
for kh in [1, 2, 3, 5]:
    m = ref.CNNMini(kh=kh, kw=kh, pad=(kh - 1) // 2, seed=0)
    h = train_model(m, X_train, y_train, X_val, y_val,
                    epochs=30, batch_size=32, lr=0.5, seed=0)
    print(f'kernel {kh}x{kh}: val acc = {h["val_acc"][-1]:.4f}')


Kernel 3x3 cukup: garis hanya butuh deteksi 'baris terang' lokal + pooling.Kernel 1x1 tetap bisa belajar (kontras garis vs latar cukup informatif),tapi kehilangan konteks tetangga; kernel besar menambah parameter tanpatambahan informasi yang berguna di gambar 8x8.

## Ketahanan terhadap pergeseran: MLP vs CNN

In [ ]:
from nn_mini.augment import random_shift

rng = np.random.default_rng(7)
for geser in [1, 2, 3]:
    Xv = random_shift(X_val, rng, max_shift=geser)
    acc_mlp = float(((mlp.predict_proba(Xv.reshape(len(Xv), -1)) >= 0.5) == y_val).mean())
    acc_cnn = float(((cnn.predict_proba(Xv) >= 0.5) == y_val).mean())
    print(f'geser ±{geser}px: MLP = {acc_mlp:.3f} | CNN = {acc_cnn:.3f}')


CNN jauh lebih tahan: konvolusi + pooling mencari POLA, bukan POSISI.MLP yang mengahmali koordinat piksel kehilangan akurasi begitu garis pindah.

## Test set (sekali, di akhir)

In [ ]:
p_test = cnn.predict_proba(X_test)
acc_test = float(((p_test >= 0.5) == y_test).mean())
print(f'TEST accuracy: {acc_test:.4f}')

for i in range(5):
    pred = int(p_test[i] >= 0.5)
    print(f'sampel {i}: prediksi={NILAI_KELAS[pred]}, benar={NILAI_KELAS[y_test[i]]}, '
          f'p={p_test[i]:.2f}', '✓' if pred == y_test[i] else '✗')
